In [1]:
import torch
import onnx
import onnxruntime as ort
import numpy as np
from glasses_detector import GlassesClassifier
from PIL import Image
glass_classifier = GlassesClassifier(kind="anyglasses", size="medium")
img = Image.open("picture.jpg")
print(glass_classifier("picture.jpg"))

present


In [2]:
import inspect
src = inspect.getsource(glass_classifier.__class__.__call__)
print(src)

    @override
    def predict(
        self,
        image: (
            FilePath
            | Image.Image
            | np.ndarray
            | Collection[FilePath | Image.Image | np.ndarray]
        ),
        format: str | dict[bool, Default] | Callable[[torch.Tensor], Default] = "str",
        input_size: tuple[int, int] | None = (256, 256),
    ) -> Default | list[Default]:
        """Predicts whether the positive class is present.

        Takes a path or multiple paths to image files or the loaded
        images themselves and outputs a formatted prediction for each
        image indicating whether it belongs to a positive class, e.g.,
        *"anyglasses"*, or not. The format of the prediction,
        i.e., the prediction type is
        :data:`~glasses_detector.components.pred_type.Default` type
        which corresponds to :attr:`~.PredType.DEFAULT`.

            If the image is provided as :class:`numpy.ndarray`, make
            sure the last dimension specifies the ch

In [3]:
import onnxruntime
model = onnxruntime.InferenceSession("glasses_classifier_single.onnx")


def preprocess(image: Image.Image) -> np.ndarray:
    img = image.resize((256, 256))
    img = np.array(img).astype(np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img = (img - mean) / std
    img = np.transpose(img, (2, 0, 1))
    img = np.expand_dims(img, axis=0)
    return img


img_np = preprocess(img)
inputs = {model.get_inputs()[0].name: img_np}
outputs = model.run(None, inputs)
outputs

[array([[0.44852865]], dtype=float32)]

In [6]:
def run_model(image: Image.Image):
    img_np = preprocess(image)
    inputs = {model.get_inputs()[0].name: img_np}
    outputs = model.run(None, inputs)
    outputs = np.array(outputs[0]).flatten()[0]
    return outputs


without_glass = Image.open("without_glass.jpg")
without_glass.resize((500, 500))
print(run_model(without_glass))

-3.2030125
